SEGMENTAZIONE SEMANTICA: L'ARCHITETTURA U-NET E LA VISIONE PIXEL-LEVEL

sappiamo insegnare ai nostri algoritmi a mettere dei rettangoli sulle nostre foto.
Ma immagina un chirugo che deve operare, o una macchina che deve passare per una strada stretta, il rettangolo potrebbe non bastare.

- L'anatomia della U-net
- Oltre le Bounding Box 
- Applicazioni SOTA: dove questa tencologia sta cambiando il mondo

Mentre l'Objected Deteciont lavora per rettangoli (dice c'è un cane in questo rettangolo) la Segmentazione Semantica lavora pixel per pixel.
Immagine un'immagine 512x512, hai 262.144 pixel (512X512=262.144). La rete deve essegnare una classe a ciascuno di quei pixel
Per esempio:
0=sfondo
1=persona
2=automobile
3=strada
L'output non è quindi una sola classe, ma una mappa della stessa dimensione spaziale dell'immagine (n. di pixel)
Nella realtà ogni posizione corrisponde ad un pixel
1111111 - cielo
1122111 - persona
3334433 - strada + auto
quindi abbiamo:
Semantic Segmentation = classificazione di ogni pixel dell'immagine.
Semantic vuol dire che tutti gli oggetti della stessa classe ricevono la stessa etichetta.
Se nell'immagine ci sono 3 persone (persona A, perosna B, persona C) la segmentazione semantica produce per 3 volte persona (persona, persona, persona), non distingue le singole identità (in quel caso parleresti di Instance Segmentation per ottenere persona #1, persona #2, persona #3)
quindi:
Classification -> cosa c'è?
Semantic Segmentation -> a quale classe appartiene ogni pixel?
Instance Segmentation -> a quele singola istanza appartiene ogni pixel?

Perchè serve U-NET?
Una normale CNN fa qualcosa del genere:
immagine -> conv -> pooling -> conv -> pooling -> ... ->classificazione
man mano che vai anvanti: 
256x256 -> 64x64 -> 32x32
riduci la dimensione spaziale.
Questo è utile per capire cosa c'è nell'immagine
Ma crea un problema enorme se devi sapere esattamente: "quali pixel appartengono all'oggetto?" perchè hai perso risoluzione.
Ed è qui  che U-Net è utile

L'Architettura U-Net
Nata nel 2015 per risolvere un problema enorme in ambito medico, avevamo pochissimi dati ed avevamo bisogno di una precisione estrema.
La U-Net estrae il significato profondo (che cosa stiamo guardando) ma allo stesso tempo si ricorda dei dettagli spaziali minimi necessari per una localizzazione precisa (dove si trovano i bordi). 

L'architettura si divede in 3 grandi fasi

Encoder, Decoder e Skip Connection
I tre pilastri della struttura a U
- Contracting Path (Encoder): una serie di convoluzioni e pooling che riducono la dimensione spaziale aumentando la profondità dei canali per catturare il contesto. L'encoder assomiglia a una CNN classica: nei primi layer riconosci bordi, linee, texture, nei layer più profondi forme, parti, strutture, oggetti. 
E' come un detective che restringe il campo, riduce l'immagine per capire il contesto, ma così facendo perde i dettagli fini
L'encoder risponde bene a: "Che cosa sto gaurdando?" ma perde progressivamente precisione sulla posizione esatta.
- Expanding Patch (Decoder): processo di up-sampling che ricostruisce la risoluzione spaziale dell'immagine partendo dalle feature estratte. Devi fare il contrario del Encoder, ricostruire progressivamente una mappa ad alta risoluzione. Cerca quindi di rispondere alla domanda: "dove esattamente si trova ciò che ho riconosciuto?". 
Qui nasce un problema, l'encoder ha buttato via i dettagli
e U-Net risonde a questa domanda  con la sua idea più importante: le skip connection
- Skip Connections: collegamenti diretti che concatenano le mappe di feature dell'encoder con quelle corrispondenti del decoder
Sono come dei ponti radio che teletrasportano l'informazione dall'inizio alla fine, senza passare per il collo di bottiglia centrale.
- Concatenazione semantica: le skip connection forniscono al decoder i dettagli perduti durante il pooling, essenziale per definire i bordi degli oggetti.

Il Flusso dei Dati nella U-Net
Partiamo da un'immagine RGB
altezza = 256, larhezza = 256, canali = 3
La U-Net fa due movimenti opposti:
ENCODER: riduce la risoluzine ed aumenta le feature
DECODER: aumenta la risoluzione, ricostruisce la segmentazione
Il BOTTLENECK si pone tra i due (encoder, decoder) è il punto più compresso della rete.
Qui l'immagine originale è stata trasformata in una rappresentazione molto ricca di feature ma povera di dettagli spaziali. Prima di perdere le informazioni spaziali le feature spaziali sono copiate dell'encoder direttamente nel decoder tramite le skip connection
Input: vedo ogni pixel
Bottleneck: ho capito cosa c'è ma ho perso precisione su dov'è
Ed è qui che entra in gioco il decoder
Il decoder fa l'operazione opposta, quindi aumenta risoluzione
Ma se facesso solo unsampling, avrebbe un problema: non riuscirebbe a recuperare bene i dettagli persi.
Per questo U-Net usa le skip connections.

Encoder: riduce lo spazio - aumenta le feature
Decoder: aumento lo spazio - riduce le feature
Skip Connection: recupera il dettaglio

La part inferiore della U rappresenta il punto di massimo compressione, dove la rete possiede una rappresentazione puramente semantica ma priva di coorinate spaziale precise (Bottleneck e Astrazione).
Non ha informazioni spaziale esatte, ma più o meno sa dire dove l'oggetto è (c'è probabilmente un oggetto qui, ma non sa esattamente dove passa il bordo)
A differenza del semplice ridimensionamento, le convoluzioni trasposte sono layer apprendibili che imparano come espandere i dati nel modo più efficiente per la ricostruzione (Transposed Convolution).
La corrispondenza esatta tra i livelli di down-sampling e up-sampling garantisce che le feature a diverse scale possano essere integrate perfettamente tramite le skip connectins (Simmetria Architetturale).

Grazie alla simmetria (la U è simmetriaca) ogni livello della risalita ha un gemello della discesa da cui può ricevere i dettagli mancanti. Quindi da una parte scende e dall'altra risale in modo simmetrico.

Cosa significa davvero classificare a livello di pixel?

Pixel-levels vs Object-level
La Granularità del Riconoscimento Visivo
Mentre l'object detection si limita a identificare la presenza e la posizione approssimativa di un oggetto (l'oggeto persona) tramite la bounding box, la segmentazione semantica assegna ogni singolo pixel a unca categoria specifica (persona, macchina, auto, ecc), e come dare ad un bambino e dirgli di colorare di blu tutto ciò che è strada, di verde tutto ciò che è persone e di giallo tutto ciò che è cielo. Non ci sono più rettangoli, ogni pixel è un identità indipendente che deve essere categorizzata.
Questo approccio trasforma l'immagine in una maschera categorica densa, permettendo di distinguere non solo dove si trova un oggetto, ma anche la sua forma esatta e i suoi confini millimetrici.

Come si trasforma l'output della U-Net?

Dense Prediction e Maschere
Dal rettangolo alla forma organica
Ogni pixel contiene la probabilità di appartenere ad una classe.
- Classificazione Densa: ogni pixel dell'immagine di output è il risultato di una funzione Softmax che determina l'appartenenza a una classe. Funzione softmax applicata pixel per pixel.
- Invarianza alla Forma: a differenza dei modelli basati su anchor box, la segmentazione può gestire oggeetti di qualsiasi forma irregolare
- Ground Thruth Binario/Categorico: i dati di addestramento sono maschere dove il valore del pixel indica direttamente la classe di appartenenza.
- One-Hot Encoding Spaziale: l'output della rete è spesso un tensore di tanti canali quante sono le classi target del problema.

Ma come facciamo a dire alla rete che ha sbagliato a colorare un bordo?
usiamo funzioni di loss specifiche

Funzioni di Loss per la Segmentazione
- Pixel-wise Cross Entropy: è la nostra frusta, punisce ogni pixel sbagliato. Viene calcolato l'errore per ogni singolo pixel e mediato su tutta l'immagine, spingendo la rete a correggeere anche le piccole imperfezioni sui bordi.
- Dice Coefficient Loss: una metrica basata sulla sovrapposizione tra la maschera predetta e quella reale, NON è molto efficace quando le classi sono sbilanciate (es. un piccolo tumore su una grande radiografica). Quando l'oggetto che cerchiamo è piccolissimo rispetto allo sfondo la rete potrebbe semplicemente dire: "è tutto sfondo" ed avere una accuratezza del 99,99%. Ecco perchè usiamo la Dice Loss che si concentra sulla sovrapposizione tra la nostra maschere e quella reale, ignorando quanto è grande il vuoto attorno.
- Softmax sull'ultimo layer: l'ultimo layer della U-Net utilizza una convoluzione 1x1 seguita da Softmax per mappare i canali delle feature nelle probabilità delle classi desiderate.

Come misuriamo oggettivamente, il successo di questa sovrapposizione?

Valutazione della Sovrapposizione
Intersection over Union per Pixel
Per valutare la bontà della segmentazione, utilizziamo la metrica IoU calcolata non più sulle aree dei rettangoli ma sul conteggio effettivo dei pixel correttamente classificati rispetto all'unione delle aree.
Questa metrica è fondamentale per capire se la rete sta sovra-segmentando o sotto-segmentando le aree di interesse.
E' il rapporto tra i pixel che abbiamo azzeccato e l'area totale coperta sia dalla nostra previsione che dalla realtà
Un IoU elevato significa che abbiamo predetto bene

Campi di Applicazione
Dalla Diagnostica Medica alla Guida Autonoma
La segmentazione semantica non è solo un esercizio accademico, ma la tecnologia alla base di sistemi di salva-vita e automazioni complesse. La precisione pixel-pixel è un requisito non negoziabili in molti settori industriali.
Analizziamo come la U-Net è i suoi derivati vengono utilizzati per interpretare scenari dinamici e dati sensoriali complessi, partendo dall'esempio pratico che vedremo nel codice.

Ma approfondiamo i due settori trainanti di questa rivoluzione

Medicina e Automotice
Precisione e sicurezza in scenari reali.
* Medical Imaging: segmentazione di organi, tumori o vasi sanguigni in scansioni CT, MRI o vetrini istologici per supporto alle chirurgia. La U-Net è lo standard assoluto, in questo campo bisogna essere esatti.
* Delineazione dei Bordi: in medicina, un errore di pochi pixel può fare la differenza tra tessuto sano e patologico rendendo la U-Net lo standard de facto.
* Self-driving Cars: identificazione della superficie stradale, dei marciapiedi, della segnaletica orizzontale e degli ostacoli mobili
* Drivable Area: la capacità di mappare esattamente dove il veicolo può transitare in sicurezza, gestendo occlusioni e ombre.

Ma quali sono gli ostacolo tecnici che rendono tutto questo così difficile?

Sfide Tecniche nella Applicazioni
- Inferenza Real-time: nelle auto a guida autonoma, la segmentazione deve avvenire in pochi millisecondi, richiedendo hardware dedicato e ottimizzato del peso della rete
- Sbilanciamento delle Classi: spesso l'oggetto da segmentare occupa meno del 1% dell'immagine; la rete deve essere istruita a non ignorare questi piccoli dettagli critici
- Dati Multispettrali: in agricoltura di precisione o monitoraggio satellitare, la segmentazione avviene su immagini a molti canali, non solo RGB, sfruttando l'architettura flessibile della U-Net

Il Dice Coefficient
Metrica per Dati Sbilanciati
Prende l'intersezione e la divide per la somma delle aree
Il coefficiente di Dice è la meterica più utilizzata in campo medico. A differenza dell'accuratezza semplice, questa misura premia la precisione e il richiamo in modo bilanciato, ignorando i pixel di background correttamente classificati.
Viene spesso utilizzato direttamente come funzione di costo per forzare la rete a concentrarsi esclusivamente sulla forma del target.


In [1]:
"""
================================================================================
IMPLEMENTAZIONE DI UNA MINI U-NET (KERAS 3 + PYTORCH BACKEND)
================================================================================
Questa implementazione dimostra l'architettura U-Net, uno standard nell'analisi 
di immagini biomediche e nella segmentazione semantica.

 ARCHITETTURA:
 1. Encoder (Contracting Path): Estrae feature semantiche riducendo la risoluzione.
 2. Bottleneck: Rappresentazione latente compressa al massimo livello di astrazione.
 3. Decoder (Expanding Path): Ricostruisce la risoluzione originale.
 4. Skip Connections: Uniscono i dettagli spaziali dell'encoder con la semantica 
    del decoder per una localizzazione precisa dei pixel.

STABILITÀ:
- Keras 3 con backend PyTorch per massima flessibilità.
- Ottimizzatore AdamW per regolarizzazione integrata dei pesi.
- Formato .keras per portabilità cross-framework.
================================================================================
"""

import os

# CONFIGURAZIONE AMBIENTE: Impostiamo il backend Keras prima di caricare il modulo.
# Keras 3 è agnostico rispetto al backend (TensorFlow, PyTorch, JAX).
os.environ["KERAS_BACKEND"] = "torch"

import keras
from keras import layers

def build_mini_unet(input_shape=(256, 256, 3), num_classes=1):
    """
    Costruisce e restituisce un modello Keras basato sulla U-Net.
    
    Parametri:
    - input_shape: Dimensione dell'immagine in ingresso (Altezza, Larghezza, Canali).
    - num_classes: Numero di maschere in output (1 per segmentazione binaria).
    """
    
    # Definizione dell'Input Layer (Punto di ingresso dei dati nella rete)
    inputs = keras.Input(shape=input_shape)

    # --- 1. ENCODER (Contracting Path) ---
    # Scopo: Ridurre la dimensione spaziale aumentando la profondità (canali).
    # Il modello impara 'COSA' è presente nell'immagine (contesto).

    # Blocco 1: Convoluzioni Doppie (Kernel 3x3, Padding Same per mantenere le dimensioni)
    # layers.Conv2D(64, ...) -> Applica 64 filtri diversi per estrarre bordi e texture.
    conv1 = layers.Conv2D(64, 3, activation="relu", padding="same")(inputs)
    conv1 = layers.Conv2D(64, 3, activation="relu", padding="same")(conv1)
    
    # Max Pooling: Riduce le dimensioni (256x256 -> 128x128). 
    # Mantiene solo le attivazioni più forti, rendendo il modello invariante a piccole traslazioni.
    pool1 = layers.MaxPooling2D(pool_size=(2, 2))(conv1)

    # --- 2. BOTTLENECK ---
    # Punto di massima compressione. Qui la rete ha una visione globale dell'immagine
    # ma ha perso molti dettagli spaziali fini.
    bottleneck = layers.Conv2D(128, 3, activation="relu", padding="same")(pool1)
    bottleneck = layers.Conv2D(128, 3, activation="relu", padding="same")(bottleneck)

    # --- 3. DECODER (Expanding Path) ---
    # Scopo: Ripristinare la dimensione spaziale (Upsampling).
    # Il modello impara 'DOVE' si trovano gli oggetti identificati.

    # UpSampling: Raddoppia la dimensione spaziale (128x128 -> 256x256).
    up1 = layers.UpSampling2D(size=(2, 2))(bottleneck)
    
    # --- 4. SKIP CONNECTION ---
    # INTERAZIONE CRITICA: Concateniamo l'output dell'Encoder (conv1) con il Decoder (up1).
    # Invece di far passare le informazioni solo attraverso il bottleneck (collo di bottiglia),
    # forniamo al decoder i dettagli spaziali perduti direttamente dal primo blocco.
    merge1 = layers.Concatenate()([conv1, up1])
    
    # Blocco Finale di Ricostruzione
    conv2 = layers.Conv2D(64, 3, activation="relu", padding="same")(merge1)
    conv2 = layers.Conv2D(64, 3, activation="relu", padding="same")(conv2)

    # --- 5. OUTPUT LAYER (Pixel-wise Classification) ---
    # Convoluzione 1x1: Mappa i 64 canali nelle classi desiderate.
    # Sigmoid: Produce un valore tra 0 e 1 per ogni pixel (Probabilità di appartenenza alla classe).
    outputs = layers.Conv2D(num_classes, 1, activation="sigmoid")(conv2)

    # Assemblaggio del Modello Finale
    model = keras.Model(inputs=inputs, outputs=outputs, name="Mini_U-Net_Professional")
    return model

# --- ESECUZIONE E COMPILAZIONE ---

# Inizializziamo il modello con i parametri di default
unet_model = build_mini_unet()

# Compilazione: 
# Optimizer AdamW: Evoluzione di Adam con una gestione del Weight Decay più corretta (Standard 2026).
# Loss Binary Crossentropy: Misura l'errore tra la maschera predetta e quella reale a livello di pixel.
# Metric IoU (Intersection over Union): La metrica regina per la segmentazione.
unet_model.compile(
    optimizer="adamw",
    loss="binary_crossentropy",
    metrics=["accuracy", keras.metrics.IoU(num_classes=2, target_class_ids=[1], name="mean_iou")]
)

# Visualizzazione dell'architettura (utile per debuggare i tensori in transito)
unet_model.summary()

# --- VALIDAZIONE E PERSISTENZA ---

# Salvataggio nel formato nativo .keras (Vivamente consigliato rispetto a .h5)
# Permette il caricamento trasparente tra diversi framework ML.
model_path = "semantic_unet_v1.keras"
unet_model.save(model_path)

print(f"\n[INFO] Architettura U-Net configurata correttamente con backend PyTorch.")
print(f"[INFO] Modello salvato con successo in: {model_path}")

Model: "Mini_U-Net_Professional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 256, 256,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 256, 256,  │      1,792 │ input_layer[0][0] │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 256, 256,  │     36,928 │ conv2d[0][0]      │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 128, 128,  │          0 │ conv2d_1[0][0]    │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 128, 128,  │     73,856 │ max_pooling2d[0]… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_3 (Conv2D)   │ (None, 128, 128,  │    147,584 │ conv2d_2[0][0]    │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ up_sampling2d       │ (None, 256, 256,  │          0 │ conv2d_3[0][0]    │
│ (UpSampling2D)      │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 256, 256,  │          0 │ conv2d_1[0][0],   │
│ (Concatenate)       │ 192)              │            │ up_sampling2d[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_4 (Conv2D)   │ (None, 256, 256,  │    110,656 │ concatenate[0][0] │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_5 (Conv2D)   │ (None, 256, 256,  │     36,928 │ conv2d_4[0][0]    │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_6 (Conv2D)   │ (None, 256, 256,  │         65 │ conv2d_5[0][0]    │
│                     │ 1)                │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 407,809 (1.56 MB)

 Trainable params: 407,809 (1.56 MB)

 Non-trainable params: 0 (0.00 B)


[INFO] Architettura U-Net configurata correttamente con backend PyTorch.
[INFO] Modello salvato con successo in: semantic_unet_v1.keras
